# Multiple dispatch vs Python OOP · **Python side**

*Python / Colab track — analogue of `2_dispatch_vs_oop.jl`.*

> **Read this first.** **Multiple dispatch is of almost no use in Python**: it is not in the
> language, it is not idiomatic, and **Python is not built for it** — its philosophy is
> *duck typing* (you call the method, you don't dispatch on types). If you want multiple
> dispatch as a **working tool**, that's **Julia**, not Python.
>
> So the point of this notebook is **not** to teach you to code this way in Python, but to
> **understand the contrast**: what Julia gives you **natively** (and compiled → fast), Python can
> only **emulate** with a third-party library (and at run time → with no speed gain).

⚠️ A **conceptual** module (a language point), not a benchmark: we illustrate what Python's OOP
does — and what it does **not** do.

## 1. OOP: the method lives *inside* the object

The method is picked from **a single** object — the one to the left of the dot (`obj.area()`).

In [ ]:
import math

class Shape: ...

class Circle(Shape):
    def __init__(self, r): self.r = r
    def area(self): return math.pi * self.r**2

class Rectangle(Shape):                       # same field names as the Julia twin: L, w
    def __init__(self, L, w): self.L, self.w = L, w
    def area(self): return self.L * self.w

shapes = [Circle(2.0), Rectangle(3.0, 4.0), Circle(1.0)]
[f.area() for f in shapes]

**Adding an operation** (say `perimeter`) forces you to **modify every class** — or to write an
external function that tests the types by hand (`isinstance`), which is clumsy and closed.

In [ ]:
def perimeter(f):
    if isinstance(f, Circle):
        return 2*math.pi*f.r
    elif isinstance(f, Rectangle):
        return 2*(f.L + f.w)
    raise TypeError(f"perimeter not defined for {type(f).__name__}")

[perimeter(f) for f in shapes]

## 2. Python's "single" dispatch: `singledispatch`

`functools.singledispatch` picks the method from the type of the **first** argument only —
a *single* dispatch, not a *multiple* one.

In [ ]:
from functools import singledispatch

@singledispatch
def area(shape): raise NotImplementedError

@area.register
def _(c: Circle): return math.pi * c.r**2

@area.register
def _(r: Rectangle): return r.L * r.w

[area(f) for f in shapes]

## 3. Where Python gets stuck: dispatching on **several** arguments

Imagine `collide(a, b)` (same name as the Julia twin): what we return depends on the **pair** of
types — `(Circle, Circle)`, `(Circle, Rectangle)`, `(Rectangle, Rectangle)`.

In OOP we would write `a.collide(b)`. But Python finds the method by looking at **only `type(a)`**
(the object to the left of the dot); `type(b)` plays **no** part in choosing the method. So we are
forced to test `b` by hand, *inside*:

```python
class Circle:
    def collide(self, b):
        if   isinstance(b, Circle):    return "two circles"
        elif isinstance(b, Rectangle): return "circle and rectangle"
        ...
```

→ cascades of `isinstance`, and you must **reopen every class** as soon as a new type shows up.

⚠️ **Python does NOT have multiple dispatch.** The standard library only offers `singledispatch`
(one argument). For multiple dispatch you need a **third-party library**, `multipledispatch`
(`pip install`) — it is not the language. We show it here **to illustrate** what Julia does
natively: you register one implementation **per combination of types**
(`@dispatch(Circle, Rectangle)`), and at call time it inspects the type of **both** arguments:

In [ ]:
try:
    from multipledispatch import dispatch
except ImportError:
    !pip -q install multipledispatch
    from multipledispatch import dispatch

@dispatch(Circle, Circle)
def collide(a, b): return "two circles"

@dispatch(Circle, Rectangle)
def collide(a, b): return "circle and rectangle"

@dispatch(Rectangle, Rectangle)
def collide(a, b): return "two rectangles"

print([collide(Circle(1.0), Circle(2.0)),
       collide(Circle(1.0), Rectangle(2.0, 3.0)),
       collide(Rectangle(1., 1.), Rectangle(2., 2.))])

# ...and the pair we never defined — exactly like the Julia twin, symmetry is NOT free:
try:
    collide(Rectangle(1., 1.), Circle(2.0))
except NotImplementedError as e:
    print("\nmissing (Rectangle, Circle) ->", type(e).__name__, ":", str(e)[:70], "…")

# Nothing says a collision is commutative, so you declare it — once:
@dispatch(Rectangle, Circle)
def collide(a, b): return collide(b, a)
print("after declaring symmetry :", collide(Rectangle(1., 1.), Circle(2.0)))

**So: does Python offer multiple dispatch? No — and what is this library good for?**

- It is **not** in the language: `multipledispatch` is an external dependency, **rarely used**
  (Python culture = duck typing: you call the method, you don't dispatch on types).
- It does help in **niche** cases: binary operations between home-grown types, or avoiding
  cascades of `isinstance` (*visitor pattern* style).
- **But it does not buy you Julia's speed.** It does its lookup **at run time** (types discovered
  at runtime → no specialization). Julia does *dispatch on types → native **compilation***. Same
  apparent feature, **opposite** mechanism — and performance.

> So we show it to **measure the gap**: *native + compiled* (Julia, fast) vs *library + runtime*
> (Python, handy but no faster). Once again the **run time vs compile time** distinction.

## 4. By the way: what about *duck typing*?

You might think "Python doesn't really dispatch, because it's *duck typed*". To be precise:

- **The absence of multiple dispatch** does NOT come from duck typing, but from the **single
  dispatch** of the object model: `a.method(b)` only consults `type(a)`. Even a strictly typed
  Python would have it.
- **Duck typing plays elsewhere.** In idiomatic Python, we **avoid** dispatching on types: we call
  the method and trust the object ("if it quacks like a duck…", EAFP style). Dispatching explicitly
  on types (above) cuts *against* Python style.
- **Link with module 1 (speed).** Since Python is dynamically / duck typed, the type of each object
  is only known **at run time**. So when dispatch does happen, it happens at run time, by lookup,
  **without specialization** — paid again on every call. Julia does the opposite: types are known
  **at compile time** (JIT) → it picks the method *and* compiles a specialized native version, once.
  The **same** mechanism delivers **correctness** (picking the right method) **and** **speed**.

> **Key distinction, dispatch side.** Type **discovered at run time** (Python → re-checked on every
> operation, slow) vs type **known at compile time** (Julia → specialized once, fast).

## Wrap-up

| | Python (OOP) | Julia (dispatch) |
|---|---|---|
| Where the method lives | **inside** the object | **outside** the types, generic function |
| Chosen from | the type of `self` (a single one) | the type of **all** the arguments |
| Multi-argument | third-party library (`multipledispatch`) | **native** |
| Typing | duck / dynamic → runtime dispatch, no specialization | types → **specialized compilation** |
| Link with speed | — | **the engine** of specialization/compilation |

> In Julia multiple dispatch is *native* and **generates specialized native code**: that is exactly
> what makes it fast. In Python it's an added design pattern, with no performance gain.